# TP02: CleaningData - Applying OOP and Design Patterns
## Complete All-in-One Lab Notebook

**Student Name:** LENG Devid  
**Student ID:** e20220888  
**Course:** I5 Advanced Programming for Data Science  

---

### Overview & Roadmap:
Each exercise in this practical work builds upon the previous one to construct a **modular, scalable CSV cleaning and transformation pipeline** using core OOP principles and Design Patterns:

1. **Exercise 1 (OOP Foundation):** `CSVReader` class with encapsulation.
2. **Exercise 2 (Strategy Pattern):** `MissingValueStrategy` and `DataCleaner` for interchangeable cleaning methods.
3. **Exercise 3 (Decorator Pattern):** `@log_action` and `@log_time` for non-intrusive logging and performance profiling.
4. **Exercise 4 (Factory Pattern):** `TransformFactory` and `DataTransform` subclasses for extensible data transformations.
5. **Exercise 5 (Template Method Pattern):** `DataPipeline` and `CSVDataPipeline` orchestrating the complete end-to-end workflow.

---

## Exercise 1: Build the CSVReader Class (OOP Foundation)

### Concept:
A `CSVReader` encapsulates all file reading operations, holding the file location and the loaded pandas DataFrame as encapsulated state.

In [1]:
import pandas as pd
import time
import functools
from abc import ABC, abstractmethod
from typing import List

class CSVReader:
    """
    Encapsulates CSV reading and previewing functionality.
    """
    def __init__(self, file_path: str):
        self.file_path = file_path
        self.data = None

    def read(self) -> pd.DataFrame:
        """Loads CSV file into pandas DataFrame."""
        print(f"[CSVReader] Loading data from: {self.file_path}")
        self.data = pd.read_csv(self.file_path)
        print(f"[CSVReader] Successfully loaded {len(self.data)} rows.")
        return self.data

    def preview(self, n: int = 5):
        """Displays top n rows of loaded data."""
        if self.data is None:
            print("[CSVReader] No data loaded yet. Please call read().")
            return None
        print(f"[CSVReader] --- Previewing top {n} rows ---")
        return self.data.head(n)

In [2]:
# Test Exercise 1
reader_test = CSVReader("sample_data.csv")
raw_data = reader_test.read()
display(reader_test.preview(4))

[CSVReader] Loading data from: sample_data.csv
[CSVReader] Successfully loaded 8 rows.
[CSVReader] --- Previewing top 4 rows ---


,id,name,age,salary,city
0,1,Alice,25.0,50000.0,New York
1,2,Bob,30.0,NaN,Paris
2,3,Charlie,NaN,70000.0,London
3,4,David,40.0,90000.0,TOKYO


---
## Exercise 2: Apply Pattern for Missing Values Handling (Strategy Pattern)

### Concept:
The Strategy Pattern provides a family of interchangeable algorithms under a common interface (`MissingValueStrategy`).
The `DataCleaner` acts as the context, delegating cleaning to whichever strategy is injected.

In [3]:
class MissingValueStrategy(ABC):
    """Abstract Strategy for handling missing data."""
    @abstractmethod
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        pass


class DropMissing(MissingValueStrategy):
    """Strategy 1: Drops rows with any missing value."""
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        print("[DropMissing] Dropping rows with missing values...")
        return df.dropna().copy()


class FillMean(MissingValueStrategy):
    """Strategy 2: Fills missing numeric values with the column mean."""
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        print("[FillMean] Imputing numeric missing values with mean...")
        cleaned = df.copy()
        numeric_cols = cleaned.select_dtypes(include=['number']).columns
        for col in numeric_cols:
            if cleaned[col].isnull().sum() > 0:
                mean_val = cleaned[col].mean()
                cleaned[col] = cleaned[col].fillna(mean_val)
                print(f"  - Imputed '{col}' with mean: {mean_val:.2f}")
        return cleaned


class FillMode(MissingValueStrategy):
    """Strategy 3: Fills missing values with the most frequent value (mode)."""
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        print("[FillMode] Imputing missing values with column mode...")
        cleaned = df.copy()
        for col in cleaned.columns:
            if cleaned[col].isnull().sum() > 0:
                mode_val = cleaned[col].mode().iloc[0]
                cleaned[col] = cleaned[col].fillna(mode_val)
                print(f"  - Imputed '{col}' with mode: '{mode_val}'")
        return cleaned


class DataCleaner:
    """Context class that applies the chosen MissingValueStrategy."""
    def __init__(self, strategy: MissingValueStrategy):
        self.strategy = strategy

    def set_strategy(self, strategy: MissingValueStrategy):
        """Allows runtime strategy switching."""
        self.strategy = strategy
        print(f"[DataCleaner] Strategy changed to: {strategy.__class__.__name__}")

    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        return self.strategy.handle(df)

In [4]:
# Test Exercise 2: Switching strategies dynamically
cleaner = DataCleaner(strategy=DropMissing())
print("1. Testing DropMissing:")
display(cleaner.clean(raw_data))

print("\n2. Testing FillMean:")
cleaner.set_strategy(FillMean())
display(cleaner.clean(raw_data))

print("\n3. Testing FillMode:")
cleaner.set_strategy(FillMode())
display(cleaner.clean(raw_data))

1. Testing DropMissing:
[DropMissing] Dropping rows with missing values...


,id,name,age,salary,city
0,1,Alice,25.0,50000.0,New York
3,4,David,40.0,90000.0,TOKYO
4,5,Alice,25.0,50000.0,New York
5,6,Eve,35.0,65000.0,paris
6,7,Frank,28.0,52000.0,london



2. Testing FillMean:
[DataCleaner] Strategy changed to: FillMean
[FillMean] Imputing numeric missing values with mean...
  - Imputed 'age' with mean: 30.50
  - Imputed 'salary' with mean: 62142.86


,id,name,age,salary,city
0,1,Alice,25.0,50000.000000,New York
1,2,Bob,30.0,62142.857143,Paris
2,3,Charlie,30.5,70000.000000,London
3,4,David,40.0,90000.000000,TOKYO
4,5,Alice,25.0,50000.000000,New York
5,6,Eve,35.0,65000.000000,paris
6,7,Frank,28.0,52000.000000,london
7,8,Grace,30.5,58000.000000,NaN



3. Testing FillMode:
[DataCleaner] Strategy changed to: FillMode
[FillMode] Imputing missing values with column mode...
  - Imputed 'age' with mode: '25.0'
  - Imputed 'salary' with mode: '50000.0'
  - Imputed 'city' with mode: 'New York'


,id,name,age,salary,city
0,1,Alice,25.0,50000.0,New York
1,2,Bob,30.0,50000.0,Paris
2,3,Charlie,25.0,70000.0,London
3,4,David,40.0,90000.0,TOKYO
4,5,Alice,25.0,50000.0,New York
5,6,Eve,35.0,65000.0,paris
6,7,Frank,28.0,52000.0,london
7,8,Grace,25.0,58000.0,New York


---
## Exercise 3: Add Decorators for Logging and Timing (Decorator Pattern)

### Concept:
Decorators wrap methods to inject functionality (logging entry/exit, calculating elapsed time) without altering the method's core logic.

In [5]:
def log_action(func):
    """Logs when a function starts and finishes."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        name = func.__name__
        print(f"[ACTION LOG] >>> START: {name}()")
        result = func(*args, **kwargs)
        print(f"[ACTION LOG] <<< FINISH: {name}()")
        return result
    return wrapper


def log_time(func):
    """Logs execution duration in seconds."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        duration = time.time() - start
        print(f"[TIME LOG]   DURATION: {func.__name__}() took {duration:.6f} seconds.")
        return result
    return wrapper

In [6]:
# Test Exercise 3: Decorated DataCleaner
class DecoratedDataCleaner:
    def __init__(self, strategy: MissingValueStrategy):
        self.strategy = strategy

    @log_action
    @log_time
    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        return self.strategy.handle(df)

dec_cleaner = DecoratedDataCleaner(strategy=FillMean())
res = dec_cleaner.clean(raw_data)

[ACTION LOG] >>> START: clean()
[FillMean] Imputing numeric missing values with mean...
  - Imputed 'age' with mean: 30.50
  - Imputed 'salary' with mean: 62142.86
[TIME LOG]   DURATION: clean() took 0.000813 seconds.
[ACTION LOG] <<< FINISH: clean()


---
## Exercise 4: Implement Factory Pattern for Data Transformations

### Concept:
`TransformFactory` encapsulates object creation. Given a string identifier (e.g., `'normalize'`), it returns the corresponding `DataTransform` instance.

In [7]:
class DataTransform(ABC):
    """Abstract base class for transformations."""
    @abstractmethod
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        pass


class NormalizeColumns(DataTransform):
    """Min-Max scaling to [0, 1] for numeric columns."""
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        print("[NormalizeColumns] Scaling numeric columns...")
        res = df.copy()
        numeric_cols = [c for c in res.select_dtypes(include=['number']).columns if c != 'id']
        for col in numeric_cols:
            mn, mx = res[col].min(), res[col].max()
            if mx > mn:
                res[col] = (res[col] - mn) / (mx - mn)
        return res


class RemoveDuplicates(DataTransform):
    """Removes duplicate records from DataFrame."""
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        print("[RemoveDuplicates] Dropping duplicate rows...")
        return df.drop_duplicates(subset=[c for c in df.columns if c != 'id']).copy()


class StandardizeText(DataTransform):
    """Standardizes string columns to lower case and trimmed text."""
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        print("[StandardizeText] Cleaning string columns...")
        res = df.copy()
        for col in res.select_dtypes(include=['object', 'string']).columns:
            res[col] = res[col].astype(str).str.strip().str.lower()
        return res


class TransformFactory:
    """Factory to produce transformation objects based on name."""
    @staticmethod
    def get_transform(transform_name: str) -> DataTransform:
        key = transform_name.lower().strip()
        if key in ["normalize", "normalize_columns"]:
            return NormalizeColumns()
        elif key in ["remove_duplicates", "deduplicate"]:
            return RemoveDuplicates()
        elif key in ["standardize_text", "clean_text"]:
            return StandardizeText()
        raise ValueError(f"Unknown transform requested: {transform_name}")

In [8]:
# Test Exercise 4: Using the Factory
factory_sample = raw_data.fillna(raw_data.mean(numeric_only=True)).fillna("unknown")

# Apply transformations via Factory
t1 = TransformFactory.get_transform("remove_duplicates")
t2 = TransformFactory.get_transform("standardize_text")
t3 = TransformFactory.get_transform("normalize")

out1 = t1.apply(factory_sample)
out2 = t2.apply(out1)
out3 = t3.apply(out2)

print("\nTransformed DataFrame:")
display(out3)

[RemoveDuplicates] Dropping duplicate rows...
[StandardizeText] Cleaning string columns...
[NormalizeColumns] Scaling numeric columns...

Transformed DataFrame:


,id,name,age,salary,city
0,1,alice,0.000000,0.000000,new york
1,2,bob,0.333333,0.303571,paris
2,3,charlie,0.366667,0.500000,london
3,4,david,1.000000,1.000000,tokyo
5,6,eve,0.666667,0.375000,paris
6,7,frank,0.200000,0.050000,london
7,8,grace,0.366667,0.200000,unknown


---
## Exercise 5: Build a Full Cleaning Pipeline (Template Method Pattern)

### Concept:
The `DataPipeline` base class establishes the **Template Method** `run()`.
The concrete `CSVDataPipeline` implements the individual steps (`load`, `clean`, `transform`, `save`) by orchestrating all the classes created in Exercises 1-4.

In [9]:
class DataPipeline(ABC):
    """
    Abstract Base Class defining the Template Method workflow.
    """
    def __init__(self):
        self.data = None

    def run(self):
        """Template Method enforcing the execution order."""
        print("=" * 60)
        print("  EXECUTING COMPLETE DATA PIPELINE (TEMPLATE METHOD)")
        print("=" * 60)
        self.load()
        self.clean()
        self.transform()
        self.save()
        print("=" * 60)
        print("  PIPELINE EXECUTION COMPLETED")
        print("=" * 60)
        return self.data

    @abstractmethod
    def load(self):
        pass

    @abstractmethod
    def clean(self):
        pass

    @abstractmethod
    def transform(self):
        pass

    @abstractmethod
    def save(self):
        pass


class FullCleaningStrategy(MissingValueStrategy):
    """Fills numeric with mean and categorical with mode."""
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        cleaned = df.copy()
        for col in cleaned.select_dtypes(include=['number']).columns:
            if cleaned[col].isnull().sum() > 0:
                cleaned[col] = cleaned[col].fillna(cleaned[col].mean())
        for col in cleaned.select_dtypes(include=['object', 'string']).columns:
            if cleaned[col].isnull().sum() > 0:
                cleaned[col] = cleaned[col].fillna(cleaned[col].mode().iloc[0])
        return cleaned


class CSVDataPipeline(DataPipeline):
    """Concrete Pipeline implementation."""
    def __init__(self, input_path: str, output_path: str,
                 clean_strategy: MissingValueStrategy,
                 transform_steps: List[str]):
        super().__init__()
        self.input_path = input_path
        self.output_path = output_path
        self.cleaner = DataCleaner(clean_strategy)
        self.transform_steps = transform_steps

    @log_action
    @log_time
    def load(self):
        reader = CSVReader(self.input_path)
        self.data = reader.read()

    @log_action
    @log_time
    def clean(self):
        self.data = self.cleaner.clean(self.data)

    @log_action
    @log_time
    def transform(self):
        for step in self.transform_steps:
            t = TransformFactory.get_transform(step)
            self.data = t.apply(self.data)

    @log_action
    @log_time
    def save(self):
        self.data.to_csv(self.output_path, index=False)
        print(f"[Save] Exported cleaned data to: {self.output_path}")

In [10]:
# Test Exercise 5: Execute the end-to-end pipeline
full_pipeline = CSVDataPipeline(
    input_path="sample_data.csv",
    output_path="cleaned_full_dataset.csv",
    clean_strategy=FullCleaningStrategy(),
    transform_steps=["remove_duplicates", "standardize_text", "normalize"]
)

final_result = full_pipeline.run()

  EXECUTING COMPLETE DATA PIPELINE (TEMPLATE METHOD)
[ACTION LOG] >>> START: load()
[CSVReader] Loading data from: sample_data.csv
[CSVReader] Successfully loaded 8 rows.
[TIME LOG]   DURATION: load() took 0.002662 seconds.
[ACTION LOG] <<< FINISH: load()
[ACTION LOG] >>> START: clean()
[TIME LOG]   DURATION: clean() took 0.000990 seconds.
[ACTION LOG] <<< FINISH: clean()
[ACTION LOG] >>> START: transform()
[RemoveDuplicates] Dropping duplicate rows...
[StandardizeText] Cleaning string columns...
[NormalizeColumns] Scaling numeric columns...
[TIME LOG]   DURATION: transform() took 0.001749 seconds.
[ACTION LOG] <<< FINISH: transform()
[ACTION LOG] >>> START: save()
[Save] Exported cleaned data to: cleaned_full_dataset.csv
[TIME LOG]   DURATION: save() took 0.001064 seconds.
[ACTION LOG] <<< FINISH: save()
  PIPELINE EXECUTION COMPLETED


In [11]:
# Final Result Inspection
print("Final Pipeline Output:")
display(final_result)

Final Pipeline Output:


,id,name,age,salary,city
0,1,alice,0.000000,0.000000,new york
1,2,bob,0.333333,0.303571,paris
2,3,charlie,0.366667,0.500000,london
3,4,david,1.000000,1.000000,tokyo
5,6,eve,0.666667,0.375000,paris
6,7,frank,0.200000,0.050000,london
7,8,grace,0.366667,0.200000,new york


### Conclusion & Key Takeaways:
- **OOP Foundation:** Encapsulated state and behaviors in clean classes (`CSVReader`).
- **Strategy Pattern:** Enabled dynamic substitution of missing value handling strategies without modifying calling code.
- **Decorator Pattern:** Added logging and timing non-intrusively adhering to the Open/Closed Principle.
- **Factory Pattern:** Centralized creation of transformation objects, promoting loose coupling.
- **Template Method Pattern:** Established a fixed skeleton for data processing workflows while giving subclasses flexibility over individual steps.